# MarketWiseAgent POC 튜토리얼

이 노트북은 `MarketWiseAgent` 코드 기반으로 실제 실행 흐름을 따라가며 프로젝트 구조를 이해하는 데 도움을 줍니다.

- `main.py`가 실행 진입점입니다.
- `agent/us_market_agent.py`가 전체 분석 파이프라인을 조율합니다.
- `utils/`는 뉴스 수집, AI 분석, 캐시, 이메일 전송, 보안 처리를 담당합니다.

## 1) 필요한 준비물

다음 항목을 준비하세요.

1. `python -m pip install -r requirements.txt`
2. `.env` 파일 생성
3. `NEWSAPI_KEY`, `OPENAI_API_KEY` 값을 등록
4. 이메일 전송 테스트를 하려면 `GMAIL_EMAIL`, `GMAIL_PASSWORD`, `GMAIL_RECIPIENT`도 설정

`.env` 예시:

```
NEWSAPI_KEY=your_newsapi_key_here
OPENAI_API_KEY=your_openai_api_key_here
GMAIL_EMAIL=youremail@gmail.com
GMAIL_PASSWORD=your_app_password_or_oauth_token
GMAIL_RECIPIENT=recipient@example.com
```

In [ ]:
import sys
print('Python 버전:', sys.version)
import pathlib
print('프로젝트 루트:', pathlib.Path('.').resolve())

## 2) 프로젝트 파일 구조 확인

주요 파일/디렉터리와 각 역할을 먼저 살펴봅니다.

In [ ]:
!find . -maxdepth 2 -type f | sort

## 3) 구성 요소 역할 정리

이 프로젝트에서 중요한 파일은 다음과 같습니다.

- `main.py`: 에이전트 실행 진입점
- `agent/us_market_agent.py`: 뉴스 수집, 정제, 분석, 리스크 평가, 보고서 생성, 저장, 이메일 전송을 순차적으로 처리
- `utils/news_fetcher.py`: NewsAPI로 미국 증시 뉴스 수집
- `utils/ai_analyzer.py`: OpenAI 기반 뉴스 분석 및 리스크/기회 평가
- `utils/semantic_cache.py`: 동일 뉴스 세트에 대한 캐시 저장/불러오기
- `utils/email_sender.py`: 결과를 이메일로 발송
- `utils/security.py`: 입력 뉴스 검증 및 프롬프트 인젝션 방어
- `config/settings.py`: 환경 변수와 기본 설정 관리

## 4) 실제 코드 흐름

`main.py`는 `USMarketNewsAgent.run()`을 호출하고, 이 메서드는 아래 순서로 실행됩니다:

1. `fetch_news()` -> `NewsCollectorAgent.collect()` -> `get_us_market_news()`
2. `analyze_news()` -> `NewsSanitizerAgent.sanitize()` -> 캐시 검사 -> AI 분석 워크플로우 실행
3. `save_result()` -> 분석 결과를 텍스트 파일로 저장
4. `print_summary()` -> 토큰 사용량 출력
5. `send_to_email_message()` -> 이메일 발송 시도

In [ ]:
from agent.us_market_agent import USMarketNewsAgent
import inspect
print(inspect.getsource(USMarketNewsAgent.run))

## 5) 환경 변수 확인

이제 필요한 환경 변수가 실제로 로드되는지 확인합니다.

In [ ]:
import os
keys = ['NEWSAPI_KEY', 'OPENAI_API_KEY', 'GMAIL_EMAIL', 'GMAIL_PASSWORD', 'GMAIL_RECIPIENT']
for key in keys:
    print(f'{key}:', bool(os.getenv(key)))

## 6) 전체 실행 테스트

이제 전체 파이프라인을 한 번 실행해 봅니다. 필요한 키가 없는 경우, 오류 메시지가 출력됩니다.

In [ ]:
from agent.us_market_agent import USMarketNewsAgent

agent = USMarketNewsAgent()
success = agent.run()
print('실행 결과:', success)

## 7) 뉴스 수집과 분석 단계 별로 확인

실제 데이터 흐름을 단계별로 확인하며 동작을 이해합니다.

In [ ]:
from agent.us_market_agent import USMarketNewsAgent

agent = USMarketNewsAgent()
print('뉴스 수집...')
collected = agent.fetch_news()
print('수집 성공:', collected)
print('기사 개수:', len(agent.articles) if agent.articles else 0)
print('첫 기사 예시:', agent.articles[0] if agent.articles else None)

## 8) 캐시 확인

같은 뉴스 세트를 다시 분석할 때 캐시가 어떻게 동작하는지 확인합니다.

In [ ]:
from utils.semantic_cache import SemanticCacheManager
from agent.us_market_agent import USMarketNewsAgent

agent = USMarketNewsAgent()
agent.fetch_news()
cache_manager = agent.cache_manager
cached = cache_manager.get_cached_report(agent.articles)
print('캐시 존재 여부:', bool(cached))
print('캐시 데이터 예시:', cached if cached else '없음')

## 9) 출력물 확인

분석 결과는 텍스트 파일로 저장됩니다. 생성된 파일을 확인합니다.

In [ ]:
!ls -1 market_analysis_* 2>/dev/null || echo '분석 결과 파일이 아직 생성되지 않았습니다.'

## 10) 코드 확장 아이디어

- `utils/news_fetcher.py`에 뉴스 소스 추가
- `utils/ai_analyzer.py`의 프롬프트를 더 정교하게 개선
- `cache/analysis_cache.json`에 메타데이터를 추가하여 날짜별 이력 관리
- 분석 결과를 웹 대시보드로 시각화
- 이메일 전송 대신 Slack/Telegram 알림 추가